In [ ]:
import os, json
import matplotlib.pyplot as plt
import pprint
import numpy as np

truncations = [
    10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 150, 200, 256
]

truncations_superpixel = [
    9, 16, 25, 36, 49, 64, 81, 100, 121, 144, 169, 196, 225, 256
]

root = '/private/home/delong/workspace/subobjects-VLM/runs/pixmo_cap/Llama-3_2-1B-dinov2_small(768px)'
splits = ['val']


excluding = ['0107-1023-superpixel_slic(225t-384px)']

def get_loss_integration(trainer_state):
    sum = 0
    for step in trainer_state['log_history']:
        sum += step['loss']
    return sum / len(trainer_state['log_history'])

tokenizer_families = [
    ('patch', 'patch_10_per_side_random'),
    ('patch', 'patch_16_per_side_random'),
    ('superpixel', 'superpixel_slic'),
    ('directsam', 'directsam_tiny_sa1b_2ep@0_1'),
]
all_results = []

for tokenizer_family, tokenizer in tokenizer_families:
    runs = os.listdir(os.path.join(root, tokenizer_family))
    for truncation in truncations if tokenizer_family != 'superpixel' else truncations_superpixel:
        for run in runs: 

            # if tokenizer_family=='patch' and f'patch_10_per_side_random(' not in run:
            #     continue
            # if tokenizer_family=='superpixel' and f'superpixel_slic(' not in run:
            #     continue
            # if tokenizer_family=='directsam' and f'directsam_tiny_sa1b_2ep@0_1(' not in run:
            #     continue
            
            if f'{truncation}t' not in run or tokenizer not in run:
                continue
                
            path = os.path.join(root, tokenizer_family, run, 'runs')
            if not os.path.exists(path):
                continue
            files = os.listdir(path)

            # print(files)

            loss_integration = None
            files.sort()
            for folder in files:
                if os.path.isdir(os.path.join(path, folder)) and folder.startswith('checkpoint-8283'):
                    trainer_state = os.path.join(path, folder, 'trainer_state.json')
                    trainer_state = json.load(open(trainer_state, 'r'))
                    loss_integration = get_loss_integration(trainer_state)
            
            if loss_integration is None:
                continue

            # for file in files:
            #     # if file.startswith('eval_') and file.endswith('.json'):
            #     if file.startswith('vlm_eval_') and file.endswith('.json'):
            #         with open(os.path.join(path, file), 'r') as f:
            #             data = json.load(f)

            all_results.append({
                'truncation': truncation,
                'tokenizer_family': tokenizer_family,
                'run': run,
                # 'average_visual_tokens': data['average_visual_tokens'],
                # 'average_loss': data['average_loss'],
                # 'perplexity': np.exp(data['average_loss']).item(),
                'loss_integration': loss_integration,

            })

print(len(all_results))


In [ ]:
import pandas as pd
df = pd.DataFrame(all_results)
print(df)
df.to_csv('all_results_truncation.csv')

In [ ]:

import seaborn as sns


plt.figure(figsize=(50, 10))
sns.lineplot(df, x="truncation", y="loss_integration", hue="tokenizer_family")
# plt.ylim(1.75, 2.25)

plt.show()

In [ ]:

font_size = 25

def visualize(all_results, y_key, y_label):
    color_offset = 10

    for split in splits:

        plt.figure(figsize=(10, 10))
        cmaps = ['Greys', 'Purples', 'Blues_r', 'Greens']

        for tokenizer_family, results in all_results.items():
            colors = plt.cm.get_cmap(cmaps.pop(0), len(results)+color_offset*2)

            previous_x, previous_y = None, None

            for i, result in enumerate(results):
                if result['evaluation_args']['split'] != split:
                    continue

                if not result['evaluation_finished']:
                    print(f"Skipping {result['evaluation_args']} because it didn't finish")
                    # remove this result
                    continue

                visual_tokenizer_config = result['checkpoint_args']['visual_tokenizer_config']
                tokenizer_name = visual_tokenizer_config.split('/')[-1].replace('.json', '')
                tokenizer_name = f"{tokenizer_name} ({result['checkpoint_args']['max_visual_tokens']})"

                plt.scatter(
                    result['average_visual_tokens'], 
                    result[y_key], 
                    label=tokenizer_name, 
                    color=colors(i+color_offset), 
                    s=300,
                    alpha=0.5
                    )
                plt.scatter(
                    result['average_visual_tokens'], 
                    result[y_key], 
                    label=tokenizer_name, 
                    color=colors(i+color_offset), 
                    s=100,
                    )
                # plt.text(result['average_visual_tokens'], result[y_key], f'{tokenizer_name}', color=colors(i+color_offset))

                if previous_x is not None:
                    plt.plot([previous_x, result['average_visual_tokens']], [previous_y, result[y_key]], color=colors(i+color_offset))

                previous_x, previous_y = result['average_visual_tokens'], result[y_key]

        # plt.title(split)
        # plt.legend(loc='lower right')
        # make x axis log scale



        plt.ylabel(y_label,  fontsize=font_size)

        plt.grid(True, which='major', linestyle='--', alpha=0.8)
        plt.grid(True, which='minor', linestyle=':', alpha=0.4)

        plt.xscale('log')
        plt.xlabel('# Tokens (log scale)',  fontsize=font_size)
        ax = plt.gca()
        xticks = [4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144, 169, 196, 225, 256]
        # skip = [49, 81, 121, 144, 196, 225]
        skip = [9, 49, 25, 81, 121, 144, 196, 225]
        xticklabels = [str(x) if x not in skip else '' for x in xticks ]
        ax.tick_params(axis='x', which='minor', bottom=False)  # Disable minor ticks


        ax.set_xticks(xticks)
        ax.set_xticklabels(xticklabels, fontsize=font_size*0.75)
        
        # plt.xticks(fontsize=font_size*0.75)
        plt.yticks(fontsize=font_size*0.75)
        # plt.ylim(1, 2)
        
        plt.show()

print(root)
visualize(all_results, y_key='loss_integration', y_label='Average Training Loss')
visualize(all_results, y_key='average_perplexity', y_label='Validation Perplexity')